In [ ]:
import pandas as pd
raw_application_df = pd.read_csv('raw_data/IS453 Group Assignment - Application Data.csv')
raw_bureau_df = pd.read_csv('raw_data/IS453 Group Assignment - Bureau Data.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
raw_bureau_df

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372202,215474,5554621,Active,currency 1,-2891,0,-1766.0,NaN,0.0,0,225000.0,0.0,0.0,0.0,Credit card,-549,NaN
372203,215474,5554624,Closed,currency 1,-2263,0,NaN,-704.0,0.0,0,0.0,0.0,0.0,0.0,Credit card,-704,NaN
372204,440726,5554632,Closed,currency 1,-870,0,-811.0,-811.0,NaN,0,44469.0,0.0,0.0,0.0,Consumer credit,-810,NaN
372205,305396,5554650,Closed,currency 1,-1745,0,-1533.0,-1592.0,0.0,0,41972.4,0.0,0.0,0.0,Consumer credit,-1592,NaN


In [ ]:
#FILTERING IN APPLICATION DATA

#filter out 'FLAG_OWN_REALTY'] == 'Y'
cleaned_application_df = raw_application_df[raw_application_df['FLAG_OWN_REALTY'] != 'Y']

#filter out people who already own a house
allowed_types = ['Rented apartment', 'With parents', 'Municipal apartment', 'Office apartment']
cleaned_application_df = cleaned_application_df[cleaned_application_df['NAME_HOUSING_TYPE'].isin(allowed_types)]

#filter out people under 21
cleaned_application_df = cleaned_application_df[cleaned_application_df['DAYS_BIRTH'] <= -7665]

# keep only those rows without missing goods price
cleaned_application_df = cleaned_application_df[cleaned_application_df['AMT_GOODS_PRICE'].notna()]

In [ ]:
# refer to: 100002

# CREDIT_ACTIVE flatten to count of TOTAL_ACTIVE and count of TOTAL_CLOSED etc. - DIRTI

# CREDIT_CURRENCY:(Assumption: currency 1 is local) -> currency 2,3,4 falls into foreign currency category and we split by counts for those columns - DIRTI
raw_bureau_df['CREDIT_CURRENCY'].value_counts()

# DAYS_CREDIT - ANSHIKA
# DAYS_CREDIT_RECENT- How recently the client borrowed (min)
# DAYS_CREDIT_OLDEST - How long client has been borrowing (max)
# RECENT_LOAN_COUNT - number of active loans within 100 days

# CREDIT_DAY_OVERDUE take average of rows to AVERAGE_CREDIT_DAY_OVERDUE  - ANSHIKA

# DAYS_CREDIT_ENDDATE take average of rows that are negative values and ACTIVE then create AVERAGE_DAYS_OVERDUE if all are closed value = 0 - ANSHIKA

# DAYS_ENDDATE_FACT - ROOPA
# 1. Loan durations
# LOAN_DURATION_SCHEDULED = DAYS_CREDIT_ENDDATE - DAYS_CREDIT
# LOAN_DURATION_ACTUAL = DAYS_ENDDATE_FACT - DAYS_CREDIT
# 2. Loan Deviation
# LOAN_DEVIATION = DAYS_ENDDATE_FACT - DAYS_CREDIT_ENDDATE: Positive → late repayment,Negative → early repayment
# MAX_RECENCY_CLOSED_LOAN	Most recent closed loan
# AVG_LOAN_DURATION_ACTUAL	Average duration of closed loans

# AMT_CREDIT_MAX_OVERDUE - ROOPA
# keep MAX_OVERDUE_AMOUNT: Max overdue = 5043.645
# COUNT_OVERDUE_LOANS: Loans with overdue > 0 = 3

# CNT_CREDIT_PROLONG: SUM_CNT_CREDIT_PROLONG - JEWEL
# - sum up how many extensions the guy asked for

# AMT_CREDIT_SUM, AMT_CREDIT_SUM_DEBT - SHINU
# - AMT_CREDIT_SUM [sum up all the rows for 1 guy] divide by AMT_CREDIT_SUM_DEBT [sum up all the rows for 1 guy] -> TOTAL_DEBT_RATIO

# AMT_CREDIT_SUM_LIMIT - JEWEL
# - TOTAL_CREDIT_LIMIT: Sum up all of the AMT_CREDIT_SUM_LIMIT

# AMT_CREDIT_SUM_OVERDUE - SHINU
# - find out the maximum of the AMT_CREDIT_SUM_OVERDUE for the active loans for the guy -> MAX_ACTIVE_AMT_CREDIT_SUM_OVERDUE -> 0 if all closed

# CREDIT_TYPE to count of TOTAL of the different types of credits. - JEWEL

# DAYS_CREDIT_UPDATE - just average all the rows and get a AVERAGE_DAYS_CREDIT_UPDATE  - NORA

# AMT_ANNUITY - NORA
# - TOTAL_ANNUITY → sum of active loan's annuities → total monthly obligations


CREDIT_CURRENCY
currency 1    1715020
currency 2       1224
currency 3        174
currency 4         10
Name: count, dtype: int64

In [ ]:
#SHINU - AMT_CREDIT_SUM_OVERDUE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,CREDIT_DAY_OVERDUE
import numpy as np

df = raw_bureau_df.copy()

g = df.groupby('SK_ID_CURR')

# max overdue on Active loans; no Active loan -> 0
amt_overdue_client = (
    df.loc[df['CREDIT_ACTIVE'] == 'Active']
    .groupby('SK_ID_CURR')['AMT_CREDIT_SUM_OVERDUE']
    .max()
)

# Map to new column
df['MAX_AMT_CREDIT_SUM_OVERDUE'] = (
    df['SK_ID_CURR']
    .map(amt_overdue_client)
    .fillna(0)
)


# Ratio uses sums per person; same value on every row for that client
sum_credit = g['AMT_CREDIT_SUM'].sum()
sum_debt = g['AMT_CREDIT_SUM_DEBT'].sum()
# total_debt_ratio = sum_credit.div(sum_debt.replace(0, np.nan))
total_debt_ratio = sum_debt.div(sum_credit.replace(0, np.nan))
df['DEBT_RATIO'] = df['SK_ID_CURR'].map(total_debt_ratio)

#taking average overdue days per person
avg_overdue = g['CREDIT_DAY_OVERDUE'].mean()
df['AVERAGE_CREDIT_DAY_OVERDUE'] = df['SK_ID_CURR'].map(avg_overdue)




In [ ]:
# JEWEL

# AMT_CREDIT_SUM_LIMIT
# - TOTAL_CREDIT_LIMIT: Sum up all of the AMT_CREDIT_SUM_LIMIT
df[df['SK_ID_CURR']==100002]
g = df.groupby('SK_ID_CURR')
total_limit = g['AMT_CREDIT_SUM_LIMIT'].sum()
df['TOTAL_CREDIT_LIMIT'] = df['SK_ID_CURR'].map(total_limit)
df[df['SK_ID_CURR']==100002]

# CNT_CREDIT_PROLONG: SUM_CNT_CREDIT_PROLONG
# - sum up how many extensions the guy asked for
g = df.groupby('SK_ID_CURR')
sum_prolong = g['CNT_CREDIT_PROLONG'].sum()
df['SUM_CNT_CREDIT_PROLONG'] = df['SK_ID_CURR'].map(sum_prolong)
df[df['SK_ID_CURR']==100002]

# CREDIT_TYPE to count of TOTAL of the different types of credits.
# Create dummies and attach directly to df
df = pd.concat([df, pd.get_dummies(df['CREDIT_TYPE'], prefix='CREDIT_TYPE')], axis=1)

# Only dummy columns (exclude original CREDIT_TYPE)
credit_type_cols = [col for col in df.columns
                    if col.startswith('CREDIT_TYPE_') and col != 'CREDIT_TYPE']

# Groupby
g = df.groupby('SK_ID_CURR')

# Map counts back
for col in credit_type_cols:
    df[col] = df['SK_ID_CURR'].map(g[col].sum())


In [ ]:
#roopa

# DAYS_ENDDATE_FACT - ROOPA
# 1. Loan durations
# LOAN_DURATION_SCHEDULED = DAYS_CREDIT_ENDDATE - DAYS_CREDIT
# LOAN_DURATION_ACTUAL = DAYS_ENDDATE_FACT - DAYS_CREDIT
# 2. Loan Deviation
# LOAN_DEVIATION = DAYS_ENDDATE_FACT - DAYS_CREDIT_ENDDATE: Positive → late repayment,Negative → early repayment
# MAX_RECENCY_CLOSED_LOAN	Most recent closed loan
# AVG_LOAN_DURATION_ACTUAL	Average duration of closed loans

df = raw_bureau_df.copy()

df["LOAN_DURATION_SCHEDULED"] = df["DAYS_CREDIT_ENDDATE"] - df["DAYS_CREDIT"]
df["LOAN_DURATION_ACTUAL"] = df["DAYS_ENDDATE_FACT"] - df["DAYS_CREDIT"]
df["LOAN_DEVIATION"] = df["DAYS_CREDIT_ENDDATE"] - df["DAYS_ENDDATE_FACT"]

# Use only closed loans for these metrics
closed_loans = df[df["CREDIT_ACTIVE"] == "Closed"].copy()
closed_group = closed_loans.groupby("SK_ID_CURR")

# User-level aggregates (closed loans)
max_recency_closed = closed_group["DAYS_ENDDATE_FACT"].max()
avg_duration_actual = closed_group["LOAN_DURATION_ACTUAL"].mean()

# Map DAYS_ENDDATE_FACT features back to bureau rows
df["MAX_RECENCY_CLOSED_LOAN"] = df["SK_ID_CURR"].map(max_recency_closed)
df["AVG_LOAN_DURATION_ACTUAL"] = df["SK_ID_CURR"].map(avg_duration_actual)

# AMT_CREDIT_MAX_OVERDUE features
overdue_group = df.assign(
    AMT_CREDIT_MAX_OVERDUE=df["AMT_CREDIT_MAX_OVERDUE"].fillna(0)
).groupby("SK_ID_CURR")

max_overdue_amount = overdue_group["AMT_CREDIT_MAX_OVERDUE"].max()
count_overdue_loans = overdue_group["AMT_CREDIT_MAX_OVERDUE"].apply(lambda s: (s > 0).sum())

# Map overdue features back to bureau rows
df["MAX_OVERDUE_AMOUNT"] = df["SK_ID_CURR"].map(max_overdue_amount)
df["COUNT_OVERDUE_LOANS"] = df["SK_ID_CURR"].map(count_overdue_loans)

,SK_ID_CURR,MAX_RECENCY_CLOSED_LOAN,AVG_LOAN_DURATION_ACTUAL,MAX_OVERDUE_AMOUNT,COUNT_OVERDUE_LOANS
0,215354,-153.0,399.6,77674.5,1
1,215354,-153.0,399.6,77674.5,1
2,215354,-153.0,399.6,77674.5,1
3,215354,-153.0,399.6,77674.5,1
4,215354,-153.0,399.6,77674.5,1


In [ ]:
# Driti - CREDIT_ACTIVE, CREDIT_CURRENCY

df = raw_bureau_df.copy()
g = df.groupby('SK_ID_CURR')

# CREDIT_ACTIVE: count of each status per client
active_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Active').sum())
closed_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Closed').sum())
sold_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Sold').sum())
bad_debt_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Bad debt').sum())

df['TOTAL_ACTIVE'] = df['SK_ID_CURR'].map(active_count)
df['TOTAL_CLOSED'] = df['SK_ID_CURR'].map(closed_count)
df['TOTAL_SOLD'] = df['SK_ID_CURR'].map(sold_count)
df['TOTAL_BAD_DEBT'] = df['SK_ID_CURR'].map(bad_debt_count)

# CREDIT_CURRENCY: local vs foreign loan counts
# Assumption: currency 1 is local, everything else is foreign
local_count = g['CREDIT_CURRENCY'].apply(lambda x: (x == 'currency 1').sum())
foreign_count = g['CREDIT_CURRENCY'].apply(lambda x: (x != 'currency 1').sum())

df['COUNT_LOCAL_CURRENCY_LOANS'] = df['SK_ID_CURR'].map(local_count)
df['COUNT_FOREIGN_CURRENCY_LOANS'] = df['SK_ID_CURR'].map(foreign_count)

# Verify with one customer
df[df['SK_ID_CURR']==215354][['SK_ID_CURR', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY',
    'TOTAL_ACTIVE', 'TOTAL_CLOSED', 'TOTAL_SOLD', 'TOTAL_BAD_DEBT',
    'COUNT_LOCAL_CURRENCY_LOANS', 'COUNT_FOREIGN_CURRENCY_LOANS']]

,SK_ID_CURR,CREDIT_ACTIVE,CREDIT_CURRENCY,TOTAL_ACTIVE,TOTAL_CLOSED,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS
0,215354,Closed,currency 1,6,5,0,0,11,0
1,215354,Active,currency 1,6,5,0,0,11,0
2,215354,Active,currency 1,6,5,0,0,11,0
3,215354,Active,currency 1,6,5,0,0,11,0
4,215354,Active,currency 1,6,5,0,0,11,0
5,215354,Active,currency 1,6,5,0,0,11,0
6,215354,Active,currency 1,6,5,0,0,11,0
225157,215354,Closed,currency 1,6,5,0,0,11,0
225158,215354,Closed,currency 1,6,5,0,0,11,0
225159,215354,Closed,currency 1,6,5,0,0,11,0


In [11]:
# ANSHIKA

g = df.groupby('SK_ID_CURR')

# DAYS_CREDIT_RECENT
df['DAYS_CREDIT_RECENT'] = df['SK_ID_CURR'].map(g['DAYS_CREDIT'].max())

# DAYS_CREDIT_OLDEST
df['DAYS_CREDIT_OLDEST'] = df['SK_ID_CURR'].map(g['DAYS_CREDIT'].min())

# AVERAGE_CREDIT_DAY_OVERDUE
df['AVERAGE_CREDIT_DAY_OVERDUE'] = df['SK_ID_CURR'].map(g['CREDIT_DAY_OVERDUE'].mean())

# RECENT_LOAN_COUNT
recent_active = df[
    (df['CREDIT_ACTIVE'] == 'Active') &
    (df['DAYS_CREDIT'] >= -100)
]

recent_loan_count = recent_active.groupby('SK_ID_CURR').size()
df['RECENT_LOAN_COUNT'] = df['SK_ID_CURR'].map(recent_loan_count).fillna(0).astype(int)

df[df['SK_ID_CURR'] == 100002]


# AVERAGE_DAYS_OVERDUE
# - take average of DAYS_CREDIT_ENDDATE where:
#   CREDIT_ACTIVE == 'Active'
#   DAYS_CREDIT_ENDDATE < 0
# - if no such rows → 0

# Step 1: filter
active_overdue = df[
    (df['CREDIT_ACTIVE'] == 'Active') &
    (df['DAYS_CREDIT_ENDDATE'] < 0)
]

# Step 2: group and average
avg_days_overdue = active_overdue.groupby('SK_ID_CURR')['DAYS_CREDIT_ENDDATE'].mean()

# Step 3: map back and fill missing with 0
df['AVERAGE_DAYS_OVERDUE'] = df['SK_ID_CURR'].map(avg_days_overdue).fillna(0)

df[df['SK_ID_CURR'] == 100002]

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,LOAN_DEVIATION,MAX_RECENCY_CLOSED_LOAN,AVG_LOAN_DURATION_ACTUAL,MAX_OVERDUE_AMOUNT,COUNT_OVERDUE_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,AVERAGE_CREDIT_DAY_OVERDUE,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE
